# 영화 정보 구조화 — OpenAI 클라이언트 버전

영화 제목 입력 → **OpenAI 내장 web_search 툴로 검색** → **Structured Output**으로 공통 `Movie` 스키마에 맞춰 구조화한다.

`responses.parse(tools=[web_search], text_format=Movie)` 한 번의 호출로 **검색과 구조화가 동시에** 일어난다.

> 사전 준비: `.env` 에 `OPENAI_API_KEY` 만 있으면 된다 (검색 키 불필요). 실행: `uv run jupyter lab`.

## 1. 환경 설정

In [1]:
from pathlib import Path
from dotenv import load_dotenv

# .env 로드를 위해 프로젝트 루트 찾기 (pyproject.toml 기준).
# rnd_onboarding 패키지는 설치돼 있어 import 경로 조작은 불필요하다.
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

TITLE = "인터스텔라"  # ← 원하는 영화 제목으로 변경
MODEL = "gpt-5.4-nano"  # web_search + structured output 지원
EFFORT = "low"          # reasoning effort (low: 빠르고 저렴)
print("ROOT:", ROOT, "| TITLE:", TITLE, "| MODEL:", MODEL, "| effort:", EFFORT)

ROOT: C:\workspace\rnd-onboarding | TITLE: 인터스텔라 | MODEL: gpt-5.4-nano | effort: low


## 2. 검색 + 구조화 (단일 호출)

`tools=[{"type": "web_search"}]` 로 모델이 직접 웹을 검색하고, `text_format=Movie` 로 결과를 `Movie` 스키마에 맞춰 돌려준다.

In [2]:
from openai import OpenAI
from rnd_onboarding import prompts
from rnd_onboarding.schemas import Movie

client = OpenAI()

msgs = prompts.render("v2", TITLE)  # findings 없음 → 모델이 web_search 로 직접 검색
resp = client.responses.parse(
    model=MODEL,
    tools=[{"type": "web_search"}],
    reasoning={"effort": EFFORT},
    input=[
        {"role": "system", "content": msgs["system"]},
        {"role": "user", "content": msgs["user"]},
    ],
    text_format=Movie,
)
movie = resp.output_parsed
print(movie.pretty())

🎬 인터스텔라 (Interstellar)  (2014-11-07)
⭐ 평점      : 8.7
🎥 감독      : 크리스토퍼 놀란
🏢 제작/배급 : Paramount Pictures (대표 제작/배급사)
👥 출연      : 매슈 맥커너히, 앤 해서웨이, 제시카 차스테인, 빌 어윈, 엘런 버스틴
📖 줄거리    : 2067년, 황폐해진 지구에서 생존이 무너져 가는 가운데 NASA는 다른 행성에서 인류의 미래를 찾기 위한 성간 임무를 준비한다. 전직 조종사인 쿠퍼는 딸 머피의 반대에도 불구하고 ‘엔듀어런스’호를 타고 출발한다. 웜홀을 통과한 뒤 초거대 블랙홀 주변의 행성계에서 새로운 거주 가능성을 향한 탐사가 이어진다.


## 3. 프롬프트 매니징 — v1 vs v2 비교

같은 영화에 프롬프트 버전만 바꿔 끼우며 출력 차이를 비교한다 (`prompts` 모듈이 버전 관리).

In [3]:
for v in prompts.list_versions():
    msgs = prompts.render(v, TITLE)
    resp = client.responses.parse(
        model=MODEL,
        tools=[{"type": "web_search"}],
        reasoning={"effort": EFFORT},
        input=[
            {"role": "system", "content": msgs["system"]},
            {"role": "user", "content": msgs["user"]},
        ],
        text_format=Movie,
    )
    print(f"=== prompt {v} ===")
    print(resp.output_parsed.model_dump_json(indent=2))
    print()

=== prompt v1 ===
{
  "title": "인터스텔라 (Interstellar)",
  "release_date": "2014-11-07",
  "production": "Legendary Pictures, Syncopy, Lynda Obst Productions (제작); Paramount Pictures, Warner Bros. (배급)",
  "director": "Christopher Nolan",
  "cast": [
    "Matthew McConaughey",
    "Anne Hathaway",
    "Jessica Chastain",
    "Michael Caine",
    "Bill Irwin"
  ],
  "synopsis": "농작물이 말라가며 황폐해진 지구에서, 전직 파일럿 쿠퍼는 인류를 구할 새로운 거주지를 찾기 위한 우주 임무에 오르게 된다. 시간의 개념이 뒤틀리는 환경 속에서 임무는 점점 더 위험해지고, 쿠퍼가 사랑하는 가족과의 시간 차는 예측하기 힘든 대가로 돌아온다. 우주는 인간의 생존과 선택, 그리고 사랑의 의미를 시험한다.",
  "rating": 8.6
}



=== prompt v2 ===
{
  "title": "인터스텔라",
  "release_date": "2014-11-05",
  "production": "Legendary Pictures",
  "director": "크리스토퍼 놀란",
  "cast": [
    "매튜 맥커너히",
    "앤 해서웨이",
    "제시카 차스테인",
    "마이클 케인",
    "매켄지 포이"
  ],
  "synopsis": "21세기 지구는 거듭되는 황폐화의 위기 속에 있으며, 주인공 쿠퍼(매튜 맥커너히)는 농장을 지키며 가족과 함께 살아간다. 어느 날 딸 머프(매켄지 포이)는 우연이 아닌 이상 징후를 발견하고, 그 패턴은 중력 교란과 좌표로 이어진다. 인류의 생존을 위해 NASA는 웜홀을 거쳐 외계 행성으로 향하는 임무를 시작하고, 쿠퍼는 그 임무에 합류한다.",
  "rating": 8.7
}



## 4. 정리

- OpenAI Responses API 는 **툴 사용(web_search) + 구조화 출력(text_format)** 을 한 호출에 융합한다.
- 반환 `resp.output_parsed` 가 곧 `Movie` 인스턴스 → 후처리 불필요.
- LangChain 버전(`02_langchain_version.ipynb`)은 같은 일을 2단계 체인으로 한다 — 비교해 볼 것.